### 문제: PiPER의 joint 각도 ($q_1,\dots,q_6$)에 따른 EE 위치를 식으로 구하고, 이를 미분해 joint 속도와 EE 선속도를 연결하는 Jacobian을 유도하기


In [1]:
import mujoco
import numpy as np
import sympy as sp

model = mujoco.MjModel.from_xml_path(
    "../../external/mujoco_menagerie/agilex_piper/scene.xml"
)

- `piper_init_pose_forward_kinematics.ipynb`에서 사용한 함수 재사용


In [2]:
def get_local_transform(model, body_name):
    body = model.body(body_name)

    rotation_flat = np.empty(9)
    mujoco.mju_quat2Mat(rotation_flat, body.quat)

    T = np.eye(4)
    T[:3, :3] = rotation_flat.reshape(3, 3)  # 3x3 rotation matrix
    T[:3, 3] = body.pos  # translation vector
    return T

In [3]:
q = sp.symbols("q1:7", real=True)
q1, q2, q3, q4, q5, q6, = q

def joint_rotation_z(theta):
    c = sp.cos(theta)
    s = sp.sin(theta)

    return sp.Matrix([
        [c, -s, 0, 0],
        [s, c, 0, 0],
        [0, 0, 1, 0],
        [0, 0, 0, 1],
    ])

- PiPER 모델에서 joint1~6의 회전축이 각각 local frame의 z축으로 정의되어있음
  ```
  <joint name="joint1" axis="0 0 1" ... />
  ```
- 따라서 joint_rotation_z 함수 하나로 구현 가능


In [4]:
A1 = sp.Matrix(get_local_transform(model, "link1"))
T_0_1 = A1 * joint_rotation_z(q1)

T_0_1

Matrix([
[ 0.999999999991*sin(q1) - 4.24264237419703e-6*cos(q1), 4.24264237419703e-6*sin(q1) + 0.999999999991*cos(q1),   0,     0],
[-4.24264237419703e-6*sin(q1) - 0.999999999991*cos(q1), 0.999999999991*sin(q1) - 4.24264237419703e-6*cos(q1),   0,     0],
[                                                    0,                                                    0, 1.0, 0.123],
[                                                    0,                                                    0,   0,   1.0]])

In [5]:
A1

Matrix([
[-4.24264237419703e-6,       0.999999999991, 0.0,   0.0],
[     -0.999999999991, -4.24264237419703e-6, 0.0,   0.0],
[                 0.0,                  0.0, 1.0, 0.123],
[                 0.0,                  0.0, 0.0,   1.0]])

- q1 회전값을 변수로 둔다
- z축을 기준으로 회전
- [0 0 0.123]이 translation vector (link 길이에 대한 단서)


In [6]:
T_0_1.subs(q1, 0)  # Substitution

Matrix([
[-4.24264237419703e-6,       0.999999999991,   0,     0],
[     -0.999999999991, -4.24264237419703e-6,   0,     0],
[                   0,                    0, 1.0, 0.123],
[                   0,                    0,   0,   1.0]])

- q1에 0을 대입해본 결과


---


### ${}^{0}T_6$ 구하기


In [7]:
A2 = sp.Matrix(get_local_transform(model, "link2"))
T_1_2 = A2 * joint_rotation_z(q2)

T_1_2

Matrix([
[-3.99999999994849e-6*sin(q2) - 3.99999999994849e-6*cos(q2), 3.99999999994849e-6*sin(q2) - 3.99999999994849e-6*cos(q2),      -0.999999999984,   0],
[     7.99993404854149e-12*sin(q2) - 0.999999999992*cos(q2),     0.999999999992*sin(q2) + 7.99993404854149e-12*cos(q2),  3.99999999994849e-6,   0],
[     0.999999999992*sin(q2) - 8.00004507084395e-12*cos(q2),     8.00004507084395e-12*sin(q2) + 0.999999999992*cos(q2), -3.99999999989298e-6,   0],
[                                                         0,                                                         0,                    0, 1.0]])

In [8]:
A3 = sp.Matrix(get_local_transform(model, "link3"))
T_2_3 = A3 * joint_rotation_z(q3)

T_2_3

Matrix([
[-0.100778725975984*sin(q3) + 0.994908864364298*cos(q3), -0.994908864364298*sin(q3) - 0.100778725975984*cos(q3),   0,  0.28358],
[ 0.994908864364298*sin(q3) + 0.100778725975984*cos(q3), -0.100778725975984*sin(q3) + 0.994908864364298*cos(q3),   0, 0.028726],
[                                                     0,                                                      0, 1.0,        0],
[                                                     0,                                                      0,   0,      1.0]])

In [9]:
A4 = sp.Matrix(get_local_transform(model, "link4"))
T_3_4 = A4 * joint_rotation_z(q4)

T_3_4

Matrix([
[ 2.89276723741683e-6*sin(q4) + 0.187087561637314*cos(q4), -0.187087561637314*sin(q4) + 2.89276723741683e-6*cos(q4),   -0.982343241577116, -0.24221],
[-1.27745377986876e-6*sin(q4) + 0.982343241581116*cos(q4), -0.982343241581116*sin(q4) - 1.27745377986876e-6*cos(q4),    0.187087561634314, 0.068514],
[   -0.999999999995*sin(q4) - 7.13697318244222e-7*cos(q4),     7.13697318244222e-7*sin(q4) - 0.999999999995*cos(q4), -3.08068605792133e-6,        0],
[                                                       0,                                                        0,                    0,      1.0]])

In [10]:
A5 = sp.Matrix(get_local_transform(model, "link5"))
T_4_5 = A5 * joint_rotation_z(q5)

T_4_5

Matrix([
[                 1.0*cos(q5),                 -1.0*sin(q5),                    0,   0],
[-4.24264237419703e-6*sin(q5), -4.24264237419703e-6*cos(q5),      -0.999999999991,   0],
[      0.999999999991*sin(q5),       0.999999999991*cos(q5), -4.24264237419703e-6,   0],
[                           0,                            0,                    0, 1.0]])

In [11]:
A6 = sp.Matrix(get_local_transform(model, "link6"))
T_5_6 = A6 * joint_rotation_z(q6)

T_5_6

Matrix([
[                -1.0*cos(q6),                  1.0*sin(q6),                   0,         0],
[-4.24264237419703e-6*sin(q6), -4.24264237419703e-6*cos(q6),      0.999999999991,     0.091],
[      0.999999999991*sin(q6),       0.999999999991*cos(q6), 4.24264237419703e-6, 0.0014165],
[                           0,                            0,                   0,       1.0]])

In [12]:
T_0_6 = T_0_1 @ T_1_2 @ T_2_3 @ T_3_4 @ T_4_5 @ T_5_6

In [13]:
T_0_6

Matrix([
[                           -1.0*(-4.24264237419703e-6*((((4.24264237419703e-6*sin(q1) + 0.999999999991*cos(q1))*(7.99993404854149e-12*sin(q2) - 0.999999999992*cos(q2)) + (0.999999999991*sin(q1) - 4.24264237419703e-6*cos(q1))*(-3.99999999994849e-6*sin(q2) - 3.99999999994849e-6*cos(q2)))*(-0.994908864364298*sin(q3) - 0.100778725975984*cos(q3)) + ((4.24264237419703e-6*sin(q1) + 0.999999999991*cos(q1))*(0.999999999992*sin(q2) + 7.99993404854149e-12*cos(q2)) + (0.999999999991*sin(q1) - 4.24264237419703e-6*cos(q1))*(3.99999999994849e-6*sin(q2) - 3.99999999994849e-6*cos(q2)))*(-0.100778725975984*sin(q3) + 0.994908864364298*cos(q3)))*(-0.982343241581116*sin(q4) - 1.27745377986876e-6*cos(q4)) + (((4.24264237419703e-6*sin(q1) + 0.999999999991*cos(q1))*(7.99993404854149e-12*sin(q2) - 0.999999999992*cos(q2)) + (0.999999999991*sin(q1) - 4.24264237419703e-6*cos(q1))*(-3.99999999994849e-6*sin(q2) - 3.99999999994849e-6*cos(q2)))*(-0.100778725975984*sin(q3) + 0.994908864364298*cos(q3)) + ((4.

In [14]:
T_0_6.shape

(4, 4)

---


### 선속도 Jacobian


In [15]:
p = T_0_6[:3, 3]  # EE position [x, y, z]
J_p = p.jacobian(q)

$$
{}^0T_6=
\begin{bmatrix}
r_{11}&r_{12}&r_{13}&\boxed{x}\\
r_{21}&r_{22}&r_{23}&\boxed{y}\\
r_{31}&r_{32}&r_{33}&\boxed{z}\\
0&0&0&1
\end{bmatrix}
$$


In [16]:
zero_angles = dict.fromkeys(q, 0)
J_p.subs(zero_angles).evalf(6)

Matrix([
[-0.00141647,  0.0804109,   0.0516849, -0.000123461,  0.00792935, 0],
[  0.0549555, -8.8262e-7, -1.78016e-6,   5.65241e-9, -3.63028e-7, 0],
[          0, -0.0549555,   -0.338535,    0.0014115,  -0.0906539, 0]])

- 모든 joint 각도가 0인 순간의 EE 선속도 Jacobian
- Joint2만 +1 rad/s로 돌리면 그 순간 EE는 world frame 기준
  - +x 방향으로 8.04 cm/s
  - y 방향으로 거의 0 (위아래로만 움직여서)
  - -z 방향으로 약 5.50 cm/s 움직인다.


---

### 각속도 Jacobian


In [17]:
T_0_2 = T_0_1 * T_1_2
T_0_3 = T_0_2 * T_2_3
T_0_4 = T_0_3 * T_3_4
T_0_5 = T_0_4 * T_4_5
T_0_6 = T_0_5 * T_5_6

$$
J_{\omega,i}={}^0R_i
\begin{bmatrix}0\\0\\1\end{bmatrix}
$$


- PiPER는 joint마다 local z축으로 회전하므로 세 번째 column을 꺼낸다


In [18]:
a1 = T_0_1[:3, 2]
a2 = T_0_2[:3, 2]
a3 = T_0_3[:3, 2]
a4 = T_0_4[:3, 2]
a5 = T_0_5[:3, 2]
a6 = T_0_6[:3, 2]

J_omega = sp.Matrix.hstack(a1, a2, a3, a4, a5, a6)

- 각 joint의 축을 추출


In [19]:
J_omega.subs(zero_angles).evalf(6)

Matrix([
[  0, 8.24264e-6, 8.24264e-6,    0.996196,  7.02292e-6,    0.996196],
[  0,        1.0,        1.0, -1.09434e-5,         1.0, -2.45815e-6],
[1.0,    -4.0e-6,    -4.0e-6,   0.0871358, -3.39027e-6,   0.0871358]])

- Joint1만 1 rad/s로 돌리면 EE의 각속도는 $\begin{bmatrix}0\\0\\1\end{bmatrix}$ rad/s
- Joint2만 1 rad/s로 돌리면 EE 각속도는 약 $\begin{bmatrix}0\\1\\0\end{bmatrix}$ rad/s
- Joint4만 1 rad/s로 돌리면 EE 각속도는 약 $\begin{bmatrix}0.996\\0\\0.087\end{bmatrix}$ rad/s


---


### 실제 Jacobian과 비교


In [22]:
data = mujoco.MjData(model)
mujoco.mj_resetData(model, data)
mujoco.mj_forward(model, data)

jac_p = np.zeros((3, model.nv))
jac_r = np.zeros((3, model.nv))

mujoco.mj_jacBody(
    model, data,
    jac_p, jac_r,
    model.body("link6").id
)

- jac_p: 선속도 Jacobian
- jac_r: 각속도 Jacobian


In [ ]:
arm_dofs = [
    int(model.joint(f"joint{i}").dofadr[0]) for i in range(1, 7)
]

J_p_mujoco = jac_p[:, arm_dofs]
J_omega_mujoco = jac_r[:, arm_dofs]

In [26]:
J_p_mujoco

array([[-1.41647440e-03,  8.04109269e-02,  5.16849269e-02,
        -1.23461464e-04,  7.92935535e-03,  0.00000000e+00],
       [ 5.49555161e-02, -8.82620531e-07, -1.78016239e-06,
         5.65240843e-09, -3.63027890e-07,  0.00000000e+00],
       [ 0.00000000e+00, -5.49555044e-02, -3.38535504e-01,
         1.41149688e-03, -9.06538765e-02,  0.00000000e+00]])

In [29]:
J_p_mujoco.shape

(3, 6)

In [27]:
J_omega_mujoco

array([[ 0.00000000e+00,  8.24264237e-06,  8.24264237e-06,
         9.96196445e-01,  7.02291705e-06,  9.96196445e-01],
       [ 0.00000000e+00,  1.00000000e+00,  1.00000000e+00,
        -1.09434340e-05,  1.00000000e+00, -2.45814925e-06],
       [ 1.00000000e+00, -4.00000000e-06, -4.00000000e-06,
         8.71357731e-02, -3.39026523e-06,  8.71357731e-02]])

In [30]:
J_omega_mujoco.shape

(3, 6)

In [ ]:
mujoco.mj_jacBody(  # calculates Jacobian
    model, data,
    jac_p, jac_r,
    model.body("link6").id
)

In [ ]:
J_p_mine = np.array(J_p.subs(zero_angles), dtype=float)
J_omega_mine = np.array(J_omega.subs(zero_angles), dtype=float)

# Compare all elements using an absolute tolerance of 1e-8.
print("Position Jacobian np.allclose:", np.allclose(J_p_mine, J_p_mujoco, atol=1e-8, rtol=0))

print("Angular Jacobian np.allclose:", np.allclose(J_omega_mine, J_omega_mujoco, atol=1e-8, rtol=0))

# Report the maximum absolute error across all elements.
print("Position Jacobian max abs error:", np.max(np.abs(J_p_mine - J_p_mujoco)))

print("Angular Jacobian max abs error:", np.max(np.abs(J_omega_mine - J_omega_mujoco)))

Position Jacobian np.allclose: True
Angular Jacobian np.allclose: True
Position Jacobian maximum absolute error: 5.551115123125783e-17
Angular Jacobian maximum absolute error: 3.3306690738754696e-16
